In [ ]:
# Cell 1 — Install dependencies
!pip -q install -U transformers datasets accelerate evaluate jiwer soundfile librosa sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.1 MB/s eta 0:00:00


In [ ]:
# Cell 2 — Drive + settings
import os
os.environ["WANDB_DISABLED"] = "true"

from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/malayalam_whisper_tiny"
os.makedirs(SAVE_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
# Cell 3 — Load Malayalam dataset
from datasets import load_dataset, Audio

raw_dataset = load_dataset(
    "google/fleurs",
    data_dir="ml_in",
    revision="refs/convert/parquet",
)

raw_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=16000))

print(raw_dataset)
print("Sample:", raw_dataset["train"][0]["transcription"][:100])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ml_in/train/0000.parquet:   0%|          | 0.00/535M [00:00<?, ?B/s]

ml_in/train/0001.parquet:   0%|          | 0.00/525M [00:00<?, ?B/s]

ml_in/train/0002.parquet:   0%|          | 0.00/530M [00:00<?, ?B/s]

ml_in/train/0003.parquet:   0%|          | 0.00/531M [00:00<?, ?B/s]

...

ml_in/test/0001.parquet:   0%|          | 0.00/327M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 3043
    })
    validation: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 418
    })
    test: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 958
    })
})
Sample: ഒസാക്കയിൽ വച്ച് ചൊവ്വാഴ്ചയാണ് അദ്ദേഹം മരണപ്പെട്ടത്


In [ ]:
# Cell 4 — Load model and processor
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "openai/whisper-tiny"

processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language="Malayalam",
    task="transcribe",
)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

model.config.forced_decoder_ids = None
model.generation_config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.suppress_tokens = []
model.config.use_cache = False

# Helps reduce memory use during training
model.gradient_checkpointing_enable()

print(f"Model parameters: {model.num_parameters()/1e6:.1f}M")

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model parameters: 37.8M


In [ ]:
# Cell 5 — Preprocessing
MAX_LABEL_LENGTH = 448

def prepare_dataset(example):
    audio = example["audio"]

    example["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_features[0]

    example["labels"] = processor.tokenizer(
        example["transcription"],
        truncation=True,
        max_length=MAX_LABEL_LENGTH,
    ).input_ids

    return example

processed_dataset = raw_dataset.map(
    prepare_dataset,
    remove_columns=raw_dataset["train"].column_names,
    num_proc=1,
)

print("Done. Train size:", len(processed_dataset["train"]))

Map (num_proc=1):   0%|          | 0/3043 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/418 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/958 [00:00<?, ? examples/s]

Done. Train size: 3043


In [ ]:
# Cell 6 — Data collator
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [ ]:
# Cell 7 — Metrics
import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    print(f"  WER: {wer:.2f}%")
    return {"wer": wer}

In [ ]:
#fix training config
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=SAVE_DIR,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=5000,

    eval_strategy="steps",
    save_strategy="steps",

    eval_steps=1000,     # 🔥 less frequent eval
    save_steps=1000,     # 🔥 less frequent saving
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    predict_with_generate=True,
    generation_max_length=128,

    fp16=True,
    gradient_checkpointing=True,

    push_to_hub=False,
    save_total_limit=2,   # 🔥 reduce file ops
    report_to="none",
)

In [ ]:
# Cell 9 — Train
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

import torch
torch.cuda.empty_cache()

print("Starting fine-tuning...")
trainer.train()

trainer.save_model(f"{SAVE_DIR}/final")
processor.save_pretrained(f"{SAVE_DIR}/final")
print("✅ Fine-tuning complete. Model saved to Drive.")

Starting fine-tuning...


Step,Training Loss,Validation Loss,Wer
1000,3.470304,0.331792,216.515679
2000,1.263975,0.192617,103.205575
3000,0.907575,0.185200,87.595819
4000,0.566847,0.195541,86.515679
5000,0.443074,0.201686,86.724739


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform

  WER: 216.52%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  WER: 86.72%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fine-tuning complete. Model saved to Drive.
